# 06 — NLP: Tokenizers and Text Pipelines (Word/Char/BPE), Padding, Masks, and Datasets

Goal: build text pipelines in pure PyTorch and understand tokenizer mechanics.

_Generated: 2026-01-25_

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. Tokenization strategies

- character
- word
- subword (BPE/WordPiece/Unigram)

This chapter includes:
- simple word tokenizer
- vocabulary with specials
- padding + attention masks via collate_fn
- educational BPE training + encoding

In [ ]:

import re
from collections import Counter

def simple_word_tokenize(text: str):
    text = text.lower().strip()
    return re.findall(r"[a-z0-9]+|[^\s\w]", text)

class Vocab:
    def __init__(self, tokens, min_freq=1, specials=("<pad>","<unk>","<bos>","<eos>")):
        counts = Counter(tokens)
        self.itos = list(specials)
        for tok, c in counts.most_common():
            if c >= min_freq and tok not in specials:
                self.itos.append(tok)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.unk_id = self.stoi["<unk>"]
        self.bos_id = self.stoi["<bos>"]
        self.eos_id = self.stoi["<eos>"]

    def encode(self, tokens, add_bos=False, add_eos=False):
        ids = []
        if add_bos: ids.append(self.bos_id)
        ids += [self.stoi.get(t, self.unk_id) for t in tokens]
        if add_eos: ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        return [self.itos[i] if 0 <= i < len(self.itos) else "<bad>" for i in ids]

corpus = [
    "Hello world!",
    "Hello PyTorch. PyTorch makes tensors and models.",
    "Tokenization turns text into tokens, then ids."
]
tokens = [t for s in corpus for t in simple_word_tokenize(s)]
vocab = Vocab(tokens)
len(vocab.itos), vocab.encode(simple_word_tokenize("Hello world!"), add_bos=True, add_eos=True)

## 2. Collation: padding and attention masks

In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self): return len(self.texts)
    def __getitem__(self, i): return self.texts[i], self.labels[i]

def collate_pad(batch, vocab: Vocab):
    texts, labels = zip(*batch)
    encoded = [vocab.encode(simple_word_tokenize(t), add_bos=True, add_eos=True) for t in texts]
    lengths = torch.tensor([len(e) for e in encoded], dtype=torch.long)
    max_len = int(lengths.max())
    input_ids = torch.full((len(encoded), max_len), vocab.pad_id, dtype=torch.long)
    attention_mask = torch.zeros((len(encoded), max_len), dtype=torch.bool)
    for i, e in enumerate(encoded):
        input_ids[i, :len(e)] = torch.tensor(e, dtype=torch.long)
        attention_mask[i, :len(e)] = True
    labels = torch.tensor(labels, dtype=torch.long)
    return input_ids.to(device), attention_mask.to(device), labels.to(device), lengths.to(device)

texts = ["I like PyTorch.", "PyTorch is fast.", "I like models.", "Tokenizers make ids."]
labels = [1, 1, 0, 0]
tds = TextDataset(texts, labels)
tdl = DataLoader(tds, batch_size=2, shuffle=True, collate_fn=lambda b: collate_pad(b, vocab))
batch = next(iter(tdl))
[tuple(x.shape) for x in batch]

## 3. Char tokenizer

In [ ]:

class CharTokenizer:
    def __init__(self, texts, specials=("<pad>","<unk>","<bos>","<eos>")):
        chars = set()
        for t in texts:
            chars.update(list(t))
        self.itos = list(specials) + sorted(chars)
        self.stoi = {c:i for i,c in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.unk_id = self.stoi["<unk>"]
        self.bos_id = self.stoi["<bos>"]
        self.eos_id = self.stoi["<eos>"]

    def encode(self, text, add_bos=False, add_eos=False):
        ids = []
        if add_bos: ids.append(self.bos_id)
        ids += [self.stoi.get(c, self.unk_id) for c in text]
        if add_eos: ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        out = []
        for i in ids:
            if 0 <= i < len(self.itos):
                out.append(self.itos[i])
        return "".join(out)

ctok = CharTokenizer(corpus)
ctok.encode("Hello", add_bos=True, add_eos=True), len(ctok.itos)

## 4. Educational BPE tokenizer

This is an instructional implementation (not optimized). Production tokenizers use specialized libraries.

In [ ]:

from collections import Counter

def bpe_get_stats(words):
    stats = Counter()
    for w, freq in words.items():
        syms = w.split()
        for i in range(len(syms)-1):
            stats[(syms[i], syms[i+1])] += freq
    return stats

def bpe_merge(pair, words):
    a, b = pair
    pat = re.compile(rf'(?<!\S){re.escape(a)}\s+{re.escape(b)}(?!\S)')
    merged = {}
    for w, freq in words.items():
        merged[pat.sub(a+b, w)] = freq
    return merged

class BPETokenizer:
    def __init__(self, vocab, merges, specials=("<pad>","<unk>","<bos>","<eos>")):
        self.specials = list(specials)
        self.itos = self.specials + sorted(vocab - set(self.specials))
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.merges = merges
        self.pad_id = self.stoi["<pad>"]
        self.unk_id = self.stoi["<unk>"]
        self.bos_id = self.stoi["<bos>"]
        self.eos_id = self.stoi["<eos>"]

    @staticmethod
    def train(texts, num_merges=100, min_freq=1):
        counts = Counter()
        for text in texts:
            for tok in simple_word_tokenize(text):
                counts[tok] += 1
        words = {}
        for w, f in counts.items():
            if f >= min_freq:
                words[" ".join(list(w)) + " </w>"] = f
        merges = []
        for _ in range(num_merges):
            stats = bpe_get_stats(words)
            if not stats: break
            best = max(stats, key=stats.get)
            merges.append(best)
            words = bpe_merge(best, words)
        vocab = set()
        for w in words.keys():
            vocab.update(w.split())
        vocab.add("</w>")
        return BPETokenizer(vocab=vocab, merges=merges)

    def encode_word(self, word):
        syms = list(word) + ["</w>"]
        for a, b in self.merges:
            i = 0
            new = []
            while i < len(syms):
                if i < len(syms)-1 and syms[i] == a and syms[i+1] == b:
                    new.append(a+b)
                    i += 2
                else:
                    new.append(syms[i]); i += 1
            syms = new
        return syms

    def encode(self, text, add_bos=False, add_eos=False):
        toks = []
        if add_bos: toks.append("<bos>")
        for w in simple_word_tokenize(text):
            toks.extend(self.encode_word(w))
        if add_eos: toks.append("<eos>")
        return [self.stoi.get(t, self.unk_id) for t in toks]

bpe = BPETokenizer.train(corpus, num_merges=50)
len(bpe.itos), bpe.encode("Hello PyTorch!", add_bos=True, add_eos=True)[:30]